In [1]:
import math
import numpy as np
import faiss
from openai import OpenAI

In [ ]:
client = OpenAI()


In [3]:
records = [
    {
        "id": "KB-001",
        "category": "accounts_receivable",
        "title": "Outstanding customer balance",
        "text": "Customer has an unpaid accounts receivable balance that remains open.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-002",
        "category": "accounts_receivable",
        "title": "Invoice still unpaid",
        "text": "Invoice INV-84729 has not been paid and is currently overdue.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-003",
        "category": "orders",
        "title": "Open sales orders",
        "text": "Customer currently has multiple open sales orders that have not yet shipped.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-004",
        "category": "payments",
        "title": "Payment application issue",
        "text": "Customer says payment was sent, but the invoice still shows as outstanding because the payment has not been applied correctly.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-005",
        "category": "collections",
        "title": "Collections follow-up",
        "text": "Accounts receivable team should contact the customer regarding past-due invoices and request payment status.",
        "source_system": "CRM"
    },
    {
        "id": "KB-006",
        "category": "payroll",
        "title": "Payroll cost by division",
        "text": "Employee payroll expense is allocated across service divisions using hours worked during the pay period.",
        "source_system": "Paycor"
    },
    {
        "id": "KB-007",
        "category": "payroll",
        "title": "PTO allocation",
        "text": "Paid time off should be distributed across service divisions based on historical work ticket hours.",
        "source_system": "Paycor"
    },
    {
        "id": "KB-008",
        "category": "integration_error",
        "title": "Oversized invoice payload",
        "text": "Invoice API request failed because an embedded image caused the payload to exceed the service size limit.",
        "source_system": "Aspire"
    },
    {
        "id": "KB-009",
        "category": "integration_error",
        "title": "API timeout",
        "text": "The upstream service did not respond before the request timeout threshold was reached.",
        "source_system": "External API"
    },
    {
        "id": "KB-010",
        "category": "data_quality",
        "title": "Duplicate transaction lines",
        "text": "Duplicate transaction line records were created because the source data did not include a unique composite identifier.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-011",
        "category": "data_quality",
        "title": "Soft-deleted accounting lines",
        "text": "Historical accounting records remain in the source with an is_deleted flag and must be excluded from current reporting.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-012",
        "category": "reporting",
        "title": "Gross margin missing",
        "text": "Gross margin is not populating correctly in the reporting model and requires troubleshooting.",
        "source_system": "Power BI"
    },
    {
        "id": "KB-013",
        "category": "reporting",
        "title": "Semantic model refresh delay",
        "text": "The reporting dataset is still refreshing, so the latest financial data may not yet be available.",
        "source_system": "Power BI"
    },
    {
        "id": "KB-014",
        "category": "entity_resolution",
        "title": "Customer name mismatch",
        "text": "The same customer appears under different names across systems, including abbreviations and alternate legal names.",
        "source_system": "Master Data"
    },
    {
        "id": "KB-015",
        "category": "entity_resolution",
        "title": "Vendor alias mapping",
        "text": "Vendor records need to be matched to a canonical entity because source systems use inconsistent naming conventions.",
        "source_system": "Master Data"
    },
    {
        "id": "KB-016",
        "category": "orders",
        "title": "Unshipped order backlog",
        "text": "Several customer orders remain open because fulfillment has not yet been completed.",
        "source_system": "ERP"
    },
    {
        "id": "KB-017",
        "category": "payments",
        "title": "Customer credit balance",
        "text": "The customer has a credit on account that can be applied against future invoices.",
        "source_system": "NetSuite"
    },
    {
        "id": "KB-018",
        "category": "general",
        "title": "Office weather",
        "text": "The weather is warm and sunny today with light winds.",
        "source_system": "Weather"
    },
    {
        "id": "KB-019",
        "category": "general",
        "title": "Dog sleeping",
        "text": "The dog is asleep on the couch near the window.",
        "source_system": "Other"
    },
    {
        "id": "KB-020",
        "category": "general",
        "title": "Lunch order",
        "text": "The team ordered sandwiches and salads for lunch.",
        "source_system": "Other"
    }
]


In [5]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = math.sqrt(sum(x * x for x in a))
    magnitude_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (magnitude_a * magnitude_b)

In [4]:
record_texts = [
    f"{record['title']}. {record['text']}"
    for record in records
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=record_texts
)

for record, item in zip(records, response.data):
    record["embedding"] = item.embedding
embedding_matrix = np.array(
    [record["embedding"] for record in records],
    dtype="float32"
)

In [5]:
print(records[0]["embedding"])

[0.02581787109375, 0.014495849609375, 0.01360321044921875, 0.03485107421875, -0.01454925537109375, 0.0007109642028808594, -0.003757476806640625, -0.0224609375, 0.0165863037109375, -0.00870513916015625, 0.031219482421875, -0.05902099609375, -0.0328369140625, -0.0251617431640625, 0.08673095703125, 0.0408935546875, -0.03277587890625, -0.016143798828125, -0.035491943359375, 0.001995086669921875, 0.02716064453125, 0.0165252685546875, 0.0178070068359375, 0.01453399658203125, -0.0010614395141601562, -0.00641632080078125, -0.045745849609375, -0.03814697265625, -0.00881195068359375, -0.053070068359375, 0.0582275390625, -0.030181884765625, -0.006183624267578125, 0.024078369140625, 0.003360748291015625, 0.0239715576171875, 0.0122222900390625, 0.0225982666015625, 0.0218963623046875, -0.0232696533203125, -0.017730712890625, -0.0247802734375, 0.02105712890625, 0.019561767578125, -0.006542205810546875, 0.051300048828125, 0.0085906982421875, -0.01227569580078125, 0.028839111328125, 0.00664138793945312

In [12]:
embedding_matrix.shape[1]

1536

In [7]:
faiss.normalize_L2(embedding_matrix)

In [23]:
dimension = embedding_matrix.shape[1]

index=faiss.IndexHNSWFlat(
    dimension,
    32,
    faiss.METRIC_INNER_PRODUCT
)

index.add(embedding_matrix)

index.hnsw.efSearch = 4


In [18]:
query = "The customer says they already paid but the invoice is still open."

query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)


query_vector = np.array(
    [query_response.data[0].embedding],
    dtype="float32"
)

faiss.normalize_L2(query_vector)

scores, indicies = index.search(
    query_vector,
    3
)

In [19]:
for score, idx in zip(scores[0], indicies[0]):
    record = records[idx]
    print(
        f"{score:.4f} | "
        f"{record['id']} | "
        f"{record['title']}"
    )


0.6359 | KB-004 | Payment application issue
0.5870 | KB-002 | Invoice still unpaid
0.5609 | KB-001 | Outstanding customer balance


In [ ]:
q = np.array(query_vector, dtype="float32")

r = np.array(record["embedding"], dtype="float32")

q = q / np.linalg.norm(q)
r = r / np.linalg.norm(r)

dot_score = np.dot(q, r)

print(f"Normalized dot product:", dot_score)

In [24]:
top_k = 3
scores, indicies = index.search(
    query_vector,
    top_k
)

In [25]:
print(scores),
print(indicies)

[[0.63589853 0.5869742  0.5608882 ]]
[[3 1 0]]


In [ ]:
for score, idx in zip(scores[0], indicies[0]):
    record = records[idx]
    print(
        f"{score:.4f} | "
        f"{record['id']} | "
        f"{record['title']}"
    )

0.6359 | KB-004 | Payment application issue
0.5870 | KB-002 | Invoice still unpaid
0.5609 | KB-001 | Outstanding customer balance


: 